# A LoKI batch: the UI

The same client API as `loki-batch.ipynb`, driven by clicking instead of by
cells: fill in a form and press a button.

## Setup

In [ ]:
import tempfile
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plopp as pp
from IPython.display import display

from ess.apps import loki
from ess.apps.batch import (
    TriggerLoop,
    apply,
    backlog,
    batch_table,
    dataset_table,
    reprocess,
    trigger_status,
)
from ess.apps.client import local
from ess.apps.rules import AsOf, Bound, Like, Lookup, LookupEntry, Rule, Selector, Template
from ess.apps.sources import FolderSource
from ess.apps.spec import as_ref, dataset_ref
from ess.reduce.spec.parameters import QEdges

journal = pd.DataFrame.from_records(
    [
        (60384, 'porous silica', 'transmission'),
        (60385, 'porous silica', 'sample'),
        (60386, 'AgBeh', 'transmission'),
        (60387, 'AgBeh', 'sample'),
        (60388, 'deuterated SDS', 'transmission'),
        (60389, 'deuterated SDS', 'sample'),
        (60392, 'empty beam', 'empty-beam'),
        (60393, 'solvent', 'background'),
        (60394, 'ISIS polymer', 'transmission'),
        (60395, 'ISIS polymer', 'sample'),
    ],
    columns=['run', 'sample', 'role'],
).set_index('run')


In [ ]:
cache = loki.cache()
root = Path(tempfile.mkdtemp(prefix='loki-batch-ui-'))
incoming = root / 'incoming'
incoming.mkdir()


def arrive(*runs):
    """Let runs arrive: link their tutorial files into the folder the source reads."""
    for run in runs:
        (path,) = cache.glob(f'{run}-*.nxs')
        (incoming / path.name).symlink_to(path)


arrive(*journal.index.drop([60394, 60395]))

client = local(
    root / 'store',
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[
        FolderSource(
            incoming,
            '*.nxs',
            identity=r'(?P<run>\d+)-.*',
            instrument='loki',
            journal=journal.to_dict('index'),
        )
    ],
)


def run_ref(number):
    return dataset_ref(instrument='loki', run=number)


def curves(label, names=None):
    """The I(Q) of every member of a batch, keyed by member or by a name for it."""
    names = names or {}
    return {
        names.get(r.request.member_key, r.request.member_key): client.output(r, 'iofq')
        for r in client.batch(label)
    }


def show(out, *values):
    """Display values inside an Output widget, replacing what was there."""
    out.clear_output()
    with out:
        for value in values:
            if value is not None:
                display(value)


center = client.run(loki.BEAM_CENTER, {'sample_run': run_ref(60387)})
shared = {
    'background_run': run_ref(60393),
    'background_transmission_run': run_ref(60392),
    'empty_beam_run': run_ref(60392),
    'direct_beam': dataset_ref(path=cache / 'direct-beam-loki-all-pixels.h5'),
    'beam_center': center.ref(),
    'q': QEdges(start=0.01, stop=0.3, num_bins=100),
}
template = Template(
    name='loki-iofq-larmor',
    spec=loki.IOFQ.id,
    params=shared,
    blanks=('sample_run', 'sample_transmission_run'),
    dataset_field='sample_run',
)
transmission_lookup = Lookup(
    name='transmission',
    entries=(
        LookupEntry(
            name='nearest-transmission',
            fills={
                'sample_transmission_run': AsOf(match={'role': Like(pattern='transmission')})
            },
        ),
    ),
)
names = {str(run_ref(n)): s for n, s in journal['sample'].items()}


def with_samples(table):
    """A batch table with the sample name of each member, keyed by dataset ref."""
    return table.join(dataset_table(client)['sample']).set_index('sample', append=True)


## The app

In [ ]:
# ---- Batch pane: pick datasets, fill a row per dataset through apply, pin edits, reduce.

shared_settings = widgets.HTML(
    '<b>Shared settings</b> &nbsp; '
    f'background run: {shared["background_run"]} &nbsp; '
    f'empty-beam run: {shared["empty_beam_run"]} &nbsp; '
    f'direct beam: {shared["direct_beam"]} &nbsp; '
    f'beam centre: {shared["beam_center"]}'
)


def options_for(role):
    """Dropdown options for a journal role: run number and sample name to a ref."""
    return [
        (f'{d.run} {d.fields.get("sample", "")}', d.ref)
        for d in sorted(client.datasets(), key=lambda d: d.run or 0)
        if d.fields.get('role') == role
    ]


def picker_options():
    """Datasets of the picked role, labelled by run, sample name, and role."""
    return [
        (f'{d.run}  {d.fields.get("sample", "")}  {d.fields.get("role", "")}', d)
        for d in sorted(client.datasets(), key=lambda d: d.run or 0)
        if d.fields.get('role') == role_picker_w.value
    ]


role_picker_w = widgets.Dropdown(
    options=sorted({d.fields['role'] for d in client.datasets()}), value='sample', description='role'
)
picker_w = widgets.SelectMultiple(options=picker_options(), description='datasets', rows=6)
role_picker_w.observe(lambda _: setattr(picker_w, 'options', picker_options()), names='value')
add_button = widgets.Button(description='Add selected', button_style='primary')
rows = {}
rows_box = widgets.VBox([])
batch_out = widgets.Output()
reduce_button = widgets.Button(description='Reduce', button_style='primary')


def make_row(dataset, transmission_default):
    return {
        'dataset': dataset,
        'name': dataset.fields.get('sample', str(dataset.ref)),
        'transmission_default': transmission_default,
        'transmission_w': widgets.Dropdown(options=options_for('transmission'), value=transmission_default),
        'override_q': widgets.Checkbox(value=False, description='override Q', indent=False),
        'q_start': widgets.FloatText(value=0.01, description='start'),
        'q_stop': widgets.FloatText(value=0.3, description='stop'),
        'q_bins': widgets.IntText(value=100, description='bins'),
        'remove': widgets.Button(description='remove', layout=widgets.Layout(width='70px')),
    }


def remove_row(key):
    def handler(_=None):
        del rows[key]
        refresh_rows()

    return handler


def row_box(key, row):
    """Lay out one row: its sample name, its fixed sample run, and its fields."""
    row['remove'].on_click(remove_row(key))
    return widgets.HBox(
        [
            widgets.Label(row['name'], layout=widgets.Layout(width='120px')),
            widgets.Label(key),
            row['transmission_w'],
            row['override_q'],
            row['q_start'],
            row['q_stop'],
            row['q_bins'],
            row['remove'],
        ]
    )


def refresh_rows():
    rows_box.children = [row_box(key, row) for key, row in rows.items()]


def add_selected(_=None):
    """Fill a request per picked dataset and turn each into an editable row."""
    selected = list(picker_w.value)
    if not selected:
        return
    group = apply(client, template, selected, lookup=transmission_lookup)
    for dataset in selected:
        key = str(dataset.ref)
        transmission_default = as_ref(group[key].params['sample_transmission_run'])
        rows[key] = make_row(dataset, transmission_default)
    refresh_rows()


def pinned_values():
    """Only the cells a person edited: the transmission run and the Q override."""
    values = {}
    for key, row in rows.items():
        entry = {}
        if row['transmission_w'].value != row['transmission_default']:
            entry['sample_transmission_run'] = row['transmission_w'].value
        if row['override_q'].value:
            entry['q'] = QEdges(
                start=row['q_start'].value, stop=row['q_stop'].value, num_bins=row['q_bins'].value
            )
        values[key] = entry
    return values


def reduce_batch(_=None):
    """Validate and submit the rows; the batch table and plot, or the errors."""
    datasets = [row['dataset'] for row in rows.values()]
    group = apply(
        client, template, datasets, pinned=pinned_values(), lookup=transmission_lookup, label='samples'
    )
    reports = {key: client.validate(request) for key, request in group.items()}
    if any(not report.ok for report in reports.values()):
        show(batch_out, pd.DataFrame({k: r.model_dump() for k, r in reports.items()}).T)
        return
    client.submit_group(group)
    table = with_samples(batch_table(client, 'samples'))
    show(batch_out, table, pp.plot(curves('samples', names), norm='log'))


add_button.on_click(add_selected)
reduce_button.on_click(reduce_batch)
batch_pane = widgets.VBox(
    [
        shared_settings,
        widgets.HBox([role_picker_w, picker_w, add_button]),
        rows_box,
        reduce_button,
        batch_out,
    ]
)

# ---- Automatic pane: a rule, its trigger loop, the backlog, and one correction.

state = {'rule': None, 'loop': None, 'rule_v2': None}

rule_name_w = widgets.Text(value='iofq-auto', description='rule name')
role_w = widgets.Dropdown(
    options=sorted({d.fields['role'] for d in client.datasets()}), value='sample', description='selector role'
)
bound_w = widgets.IntText(value=60393, description='bound run')
enable_button = widgets.Button(description='Enable', button_style='primary')
arrive_button = widgets.Button(description='Arrive 60394, 60395')
check_button = widgets.Button(description='Check now')
backlog_button = widgets.Button(description='Reduce backlog', button_style='primary')
member_w = widgets.Dropdown(options=[], description='member')
q_start_w = widgets.FloatText(value=0.005, description='start')
q_stop_w = widgets.FloatText(value=0.3, description='stop')
q_bins_w = widgets.IntText(value=100, description='bins')
reduce_member_button = widgets.Button(description='Reduce member', button_style='primary')
status_out = widgets.Output()
batch_status_out = widgets.Output()


def rule_members():
    """The rule's current members, as (sample name, dataset) pairs for the dropdown."""
    keys = {str(r.request.member_key) for r in client.batch(state['rule'].name)}
    return [(d.fields.get('sample', str(d.ref)), d) for d in client.datasets() if str(d.ref) in keys]


def trigger_table():
    """The trigger status of every dataset the source knows, for the active rule."""
    rows = [
        {'dataset': str(d.ref), **trigger_status(client, state['rule'], d).model_dump()}
        for d in client.datasets()
    ]
    return dataset_table(client).join(pd.DataFrame(rows).set_index('dataset'))


def refresh_automatic():
    """Redraw the trigger status, the rule's batch, and the member picker."""
    show(status_out, trigger_table())
    show(batch_status_out, with_samples(batch_table(client, state['rule'])))
    member_w.options = rule_members()


def enable_rule(_=None):
    """Create the rule and its trigger loop, and show the trigger status."""
    state['rule'] = Rule(
        name=rule_name_w.value,
        template=template,
        lookup=transmission_lookup,
        selector=Selector(match={'role': Like(pattern=role_w.value)}, after=Bound(run=bound_w.value)),
    )
    state['loop'] = TriggerLoop(client, state['rule'])
    refresh_automatic()


def arrive_polymer(_=None):
    arrive(60394, 60395)
    refresh_automatic()


def check_now(_=None):
    """Run the trigger loop once and redraw; a second press fires on nothing."""
    fired = state['loop'].run_once()
    refresh_automatic()
    return fired


def reduce_backlog(_=None):
    client.submit_group(backlog(client, state['rule']))
    refresh_automatic()


def reduce_member(_=None):
    """Pin a Q override for one member of the rule's batch, and reduce it."""
    member = member_w.value
    q = QEdges(start=q_start_w.value, stop=q_stop_w.value, num_bins=q_bins_w.value)
    client.submit_group(apply(client, state['rule'], [member], {str(member.ref): {'q': q}}))
    refresh_automatic()


enable_button.on_click(enable_rule)
arrive_button.on_click(arrive_polymer)
check_button.on_click(check_now)
backlog_button.on_click(reduce_backlog)
reduce_member_button.on_click(reduce_member)

automatic_pane = widgets.VBox(
    [
        widgets.HBox([rule_name_w, role_w, bound_w, enable_button]),
        widgets.Label('lookup: nearest transmission before the member'),
        widgets.HBox([arrive_button, check_button, backlog_button]),
        widgets.HBox([member_w, q_start_w, q_stop_w, q_bins_w, reduce_member_button]),
        status_out,
        batch_status_out,
    ]
)

# ---- Revise pane: move a default, save a new rule version, reprocess.

bins_w = widgets.IntText(value=200, description='default Q bins')
save_button = widgets.Button(description='Save as new version')
reprocess_button = widgets.Button(description='Reprocess', button_style='primary')
revise_out = widgets.Output()


def current_rule():
    return state['rule_v2'] or state['rule']


def history_table(member):
    """Every record made for `member` under the rule: which version, what pinned."""
    records = client.records(label=state['rule'].name, member_key=str(member.ref))
    rows = [
        {
            'record': r.id,
            'rule': r.request.origin.rule,
            'pinned': list(r.request.origin.pinned),
            'q_start': r.resolved_params['q']['start'],
            'q_bins': r.resolved_params['q']['num_bins'],
        }
        for r in records
    ]
    return pd.DataFrame(rows)


def refresh_revise():
    show(
        revise_out,
        with_samples(batch_table(client, current_rule())),
        history_table(member_w.value) if member_w.value else None,
        pp.plot(curves(state['rule'].name, names), norm='log') if state['rule'] else None,
    )


def save_new_version(_=None):
    """A new template and rule version, from the form's default Q bins."""
    template_v2 = template.revise(q=QEdges(start=0.01, stop=0.3, num_bins=bins_w.value))
    state['rule_v2'] = state['rule'].revise(template=template_v2)
    refresh_revise()


def reprocess_rule(_=None):
    client.submit_group(reprocess(client, state['rule_v2']))
    refresh_revise()


save_button.on_click(save_new_version)
reprocess_button.on_click(reprocess_rule)
revise_pane = widgets.VBox(
    [bins_w, widgets.HBox([save_button, reprocess_button]), revise_out]
)

tab = widgets.Tab(children=[batch_pane, automatic_pane, revise_pane])
tab.set_title(0, 'Batch')
tab.set_title(1, 'Automatic')
tab.set_title(2, 'Revise')
display(tab)
